In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data" / "clean").exists() else Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

# 1. 기업개요_최종: 회사 마스터 테이블 (crno가 유니크 키)
corp = pd.read_csv(CLEAN_DIR / "기업개요_최종.csv", dtype=str, encoding="utf-8-sig")
corp = corp.drop(columns=["Unnamed: 0"])
master = corp.set_index("crno")
MASTER_COLS = ["corpNm", "corpEnsnNm", "enpBsadr", "enpEstbDt", "상장여부"]

afil = pd.read_csv(CLEAN_DIR / "계열회사_전처리.csv", dtype=str, encoding="utf-8-sig")
sub = pd.read_csv(CLEAN_DIR / "종속기업_정리.csv", dtype=str, encoding="utf-8-sig")

# 이름 보강용 통합 lookup: 기업개요_최종(우선) + 계열회사_전처리의 afilCmpyCrno->afilCmpyNm(대체)
name_map = master["corpNm"].to_dict()
fallback_names = (
    afil.dropna(subset=["afilCmpyCrno", "afilCmpyNm"])
    .drop_duplicates("afilCmpyCrno")
    .set_index("afilCmpyCrno")["afilCmpyNm"]
)
for crno, name in fallback_names.items():
    name_map.setdefault(crno, name)

def attach_master(df, id_col, prefix):
    sub_master = master[MASTER_COLS].add_prefix(f"{prefix}_")
    out = df.merge(sub_master, left_on=id_col, right_index=True, how="left")
    had_official = out[f"{prefix}_corpNm"].notna()
    fb = out[id_col].map(name_map)
    out[f"{prefix}_corpNm"] = out[f"{prefix}_corpNm"].fillna(fb)
    out[f"{prefix}_name_source"] = pd.NA
    out.loc[had_official, f"{prefix}_name_source"] = "기업개요_최종"
    out.loc[(~had_official) & out[f"{prefix}_corpNm"].notna(), f"{prefix}_name_source"] = "계열회사_전처리(fallback)"
    return out

# 2. 계열회사_전처리: crno(모기업) - afilCmpyCrno(계열사), 양쪽 다 ID 보유 -> 양쪽 다 ID 매칭
afil_out = afil.copy()
afil_out["relation_type"] = "계열회사"
afil_out = afil_out.rename(columns={"crno": "source_crno", "afilCmpyCrno": "target_crno", "afilCmpyNm": "target_name_raw"})
afil_out = attach_master(afil_out, "source_crno", "source")
afil_out = attach_master(afil_out, "target_crno", "target")

# 계열망(계열회사_전처리에 한 번이라도 등장하는 crno) 소속 여부
affiliate_network_ids = set(afil["crno"]) | set(afil["afilCmpyCrno"])

# 3. 종속기업_정리: crno(모기업)만 ID 보유, 종속기업 자체 ID는 없음 -> 모기업 쪽만 ID 매칭 + 이름 보강
sub_out = sub.copy()
sub_out["relation_type"] = "종속기업"
sub_out = sub_out.rename(columns={"crno": "source_crno", "sbrdEnpNm": "target_name_raw"})
sub_out["target_crno"] = pd.NA
sub_out = attach_master(sub_out, "source_crno", "source")

# 계열사 관계망을 거치지 않고 바로 종속기업으로 이어지는 경우 표시
sub_out["direct_to_subsidiary"] = ~sub_out["source_crno"].isin(affiliate_network_ids)

# 4. 공통 스키마로 통합 (세로 결합)
common_cols = [
    "relation_type", "source_crno", "source_corpNm", "source_name_source", "source_corpEnsnNm", "source_enpBsadr", "source_enpEstbDt", "source_상장여부",
    "target_crno", "target_name_raw", "target_corpNm", "target_name_source", "target_corpEnsnNm", "target_enpBsadr", "target_enpEstbDt", "target_상장여부",
    "direct_to_subsidiary",
]

afil_final = afil_out.reindex(columns=common_cols)
sub_final = sub_out.reindex(columns=common_cols)
sub_final["sbrdEnpMainBizCtt"] = sub_out["sbrdEnpMainBizCtt"].values
sub_final["domestic"] = sub_out["domestic"].values
sub_final["name_norm"] = sub_out["name_norm"].values

merged = pd.concat([afil_final, sub_final], ignore_index=True, sort=False)

merged.to_csv(CLEAN_DIR / "지식그래프_통합_ID매칭.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

# 0. 원본 로드
corp = pd.read_csv("data/clean/기업개요_최종.csv", dtype=str, encoding="utf-8-sig").drop(columns=["Unnamed: 0"])
master = corp.set_index("crno")
afil = pd.read_csv("data/clean/계열회사_전처리.csv", dtype=str, encoding="utf-8-sig")
sub = pd.read_csv("data/clean/종속기업_정리.csv", dtype=str, encoding="utf-8-sig")

# 이름 lookup: 기업개요_최종 우선, 계열회사_전처리(afilCmpyCrno->afilCmpyNm) fallback
name_map = master["corpNm"].to_dict()
fb = afil.dropna(subset=["afilCmpyCrno", "afilCmpyNm"]).drop_duplicates("afilCmpyCrno").set_index("afilCmpyCrno")["afilCmpyNm"]
for c, n in fb.items():
    name_map.setdefault(c, n)

def nm(crno):
    return name_map.get(crno)

# 계열망(계열회사_전처리에 한 번이라도 등장하는 crno)
affiliate_network_ids = set(afil["crno"]) | set(afil["afilCmpyCrno"])

# 어떤 계열사(afilCmpyCrno)를 누가(crno) 계열사로 등록했는지 -> 역방향 lookup (모기업 후보들)
parents_of = afil.groupby("afilCmpyCrno")["crno"].apply(lambda s: sorted(set(s))).to_dict()

# ===== 경우1: 모기업 - 계열회사 =====
case1 = pd.DataFrame({
    "case": 1,
    "top_crno": afil["crno"],
    "top_corpNm": afil["crno"].map(nm),
    "affiliate_crno": afil["afilCmpyCrno"],
    "affiliate_corpNm": afil["afilCmpyCrno"].map(nm),
    "subsidiary_name": pd.NA,
    "sbrdEnpMainBizCtt": pd.NA,
    "domestic": pd.NA,
    "name_norm": pd.NA,
})

# ===== 경우2: 모기업 - 종속회사 (계열망 밖, 직접 연결) =====
mask_direct = ~sub["crno"].isin(affiliate_network_ids)
sub2 = sub[mask_direct]
case2 = pd.DataFrame({
    "case": 2,
    "top_crno": sub2["crno"],
    "top_corpNm": sub2["crno"].map(nm),
    "affiliate_crno": pd.NA,
    "affiliate_corpNm": pd.NA,
    "subsidiary_name": sub2["sbrdEnpNm"],
    "sbrdEnpMainBizCtt": sub2["sbrdEnpMainBizCtt"],
    "domestic": sub2["domestic"],
    "name_norm": sub2["name_norm"],
})

# ===== 경우3: 모기업 - 계열회사 - 종속회사 (종속회사의 모기업 자체가 계열망 소속) =====
mask_chain = sub["crno"].isin(affiliate_network_ids)
sub3 = sub[mask_chain]
top_list = sub3["crno"].map(lambda c: parents_of.get(c, []))
case3 = pd.DataFrame({
    "case": 3,
    "top_crno": top_list.map(lambda l: ";".join(l) if l else pd.NA),
    "top_corpNm": top_list.map(lambda l: ";".join(dict.fromkeys(nm(c) or c for c in l)) if l else pd.NA),
    "affiliate_crno": sub3["crno"],
    "affiliate_corpNm": sub3["crno"].map(nm),
    "subsidiary_name": sub3["sbrdEnpNm"],
    "sbrdEnpMainBizCtt": sub3["sbrdEnpMainBizCtt"],
    "domestic": sub3["domestic"],
    "name_norm": sub3["name_norm"],
})

merged = pd.concat([case1, case2, case3], ignore_index=True)
merged.to_csv("data/clean/모기업_계열사_종속기업_통합.csv", index=False, encoding="utf-8-sig")